# Notebook 05 - Full RAG Pipeline

## What This Notebook Does

This notebook connects everything from the previous four notebooks into one end-to-end pipeline.

The flow:

```
Text file
  -> chunk into smaller pieces
  -> embed each chunk with sentence-transformers
  -> store chunks + embeddings in ChromaDB
  -> accept a natural language query
  -> embed the query
  -> retrieve the top-k most similar chunks from ChromaDB
  -> display the results
```

This is the Retrieval half of RAG. The Generation half (feeding retrieved chunks
to an LLM) is the next class.


## The Two-Phase Architecture

### Phase 1: Ingestion (run once, or when documents change)

Load documents -> chunk -> embed -> store in vector database.

This phase is expensive (takes time, uses compute). You only run it when
your document set changes. The result is saved to disk.

### Phase 2: Retrieval (run on every user query)

Take a user's question -> embed the question -> query the vector database
-> return top-k matching chunks.

This phase is fast. Embedding a single query and searching a few thousand
vectors takes milliseconds.

In [ ]:
# Install if needed
# !pip install sentence-transformers chromadb faiss-cpu numpy

In [1]:
import sys
sys.path.append('..')

import chromadb

from utils.embedder  import embed, embed_batch
from utils.chunker   import sentence_chunks
from utils.searcher  import chroma_add, chroma_query, print_chroma_results

d:\Naveena Natarajan\Tutor\Abhishek-Bangalore-Python\Abhishek_Implementation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




## Phase 1 - Ingestion

### Step 1: Load Documents

In [2]:
# Load both sample documents
sources = [
    ("../data/sample_sales_notes.txt", "sales_notes"),
    ("../data/sample_faq.txt",         "product_faq"),
]

loaded = []
for path, source_name in sources:
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    loaded.append({"text": text, "source": source_name, "path": path})
    print(f"Loaded: {path} ({len(text)} characters)")

Loaded: ../data/sample_sales_notes.txt (3696 characters)
Loaded: ../data/sample_faq.txt (3122 characters)


### Step 2: Chunk Each Document

In [3]:
all_chunks = []  # list of dicts: {text, source, chunk_index}

for doc in loaded:
    chunks = sentence_chunks(doc["text"], max_size=500)
    for i, chunk_text in enumerate(chunks):
        all_chunks.append({
            "text":        chunk_text,
            "source":      doc["source"],
            "chunk_index": i,
            "id":          f"{doc['source']}_chunk_{i}"
        })
    print(f"{doc['source']}: {len(chunks)} chunks created")

print()
print(f"Total chunks: {len(all_chunks)}")

sales_notes: 9 chunks created
product_faq: 7 chunks created

Total chunks: 16


### Step 3: Embed All Chunks

In [4]:
texts = [chunk["text"] for chunk in all_chunks]

print("Embedding all chunks...")
all_embeddings = embed_batch(texts)  # shape: (N, 384)

print(f"Done. Embeddings shape: {all_embeddings.shape}")

Embedding all chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.14it/s]

Done. Embeddings shape: (16, 384)


### Step 4: Store in ChromaDB

In [5]:
# Use a persistent client so the database survives restarts
client = chromadb.PersistentClient(path="../data/rag_db")

# Delete and recreate the collection on each ingestion
# (so re-running this cell does not create duplicate entries)
try:
    client.delete_collection(name="knowledge_base")
except Exception:
    pass

collection = client.get_or_create_collection(name="knowledge_base")

chroma_add(
    collection=collection,
    ids=[chunk["id"] for chunk in all_chunks],
    texts=texts,
    embeddings=all_embeddings.tolist(),
    metadatas=[
        {"source": chunk["source"], "chunk_index": chunk["chunk_index"]}
        for chunk in all_chunks
    ]
)

print(f"Stored {collection.count()} chunks in ChromaDB.")
print("Database saved to: ../data/rag_db/")

Stored 16 chunks in ChromaDB.
Database saved to: ../data/rag_db/




## Phase 2 - Retrieval

Ingestion is complete. Now we retrieve.

The database is loaded from disk - you do not need to re-run Phase 1
unless your documents change.

### Define the retrieval function

In [6]:
def retrieve(query: str, n_results: int = 3, source_filter: str = None):
    """
    Embed the query and return the top-n matching chunks from ChromaDB.

    query         : the user's question in plain text
    n_results     : how many chunks to return
    source_filter : optional — restrict search to one source document
    """
    query_embedding = embed(query).tolist()

    kwargs = dict(query_embeddings=[query_embedding], n_results=n_results)
    if source_filter:
        kwargs["where"] = {"source": source_filter}

    return collection.query(**kwargs)

### Query 1: Price Objections

In [7]:
results = retrieve("how to handle price objections", n_results=3)

print("Query: how to handle price objections")
print()
print_chroma_results(results)

Query: how to handle price objections

Result 1  (similarity: 0.6505)
  Metadata : {'chunk_index': 0, 'source': 'sales_notes'}
  Text     : SALES TRAINING NOTES — HANDLING OBJECTIONS AND CLOSING DEALS

Section 1: Price Objections

When a prospect says your product is too expensive, the ins...

Result 2  (similarity: -0.1825)
  Metadata : {'chunk_index': 5, 'source': 'sales_notes'}
  Text     : Section 4: Handling Competition

When a prospect mentions a competitor, do not attack the competitor. It looks desperate and unprofessional. Instead, ...

Result 3  (similarity: -0.1965)
  Metadata : {'chunk_index': 1, 'source': 'sales_notes'}
  Text     : The most effective technique is to reframe from price to ROI. If your product saves a team 10 hours per week at 500 rupees per hour, that is 5000 rupe...



### Query 2: Discovery Calls

In [8]:
results = retrieve("what questions should I ask during a discovery call", n_results=3)

print("Query: what questions should I ask during a discovery call")
print()
print_chroma_results(results)

Query: what questions should I ask during a discovery call

Result 1  (similarity: 0.3531)
  Metadata : {'chunk_index': 2, 'source': 'sales_notes'}
  Text     : Section 2: Discovery Questions

Before you pitch anything, you need to understand the prospect's situation. Most salespeople talk too much in discover...

Result 2  (similarity: -0.2513)
  Metadata : {'chunk_index': 4, 'source': 'sales_notes'}
  Text     : "If we could get this set up in two weeks, does that timeline work for your team?"

If they say yes, the close is already done. If they say no, you le...

Result 3  (similarity: -0.3449)
  Metadata : {'chunk_index': 3, 'source': 'sales_notes'}
  Text     : Situation: "What does your current process for handling invoicing look like?"
Problem: "Where does that process break down most often?"
Implication: "...



### Query 3: How Does the RAG System Work - Searching the FAQ

In [9]:
# Restrict to FAQ only
results = retrieve(
    "what happens when the answer is not in the documents",
    n_results=2,
    source_filter="product_faq"
)

print("Query (FAQ only): what happens when the answer is not in the documents")
print()
print_chroma_results(results)

Query (FAQ only): what happens when the answer is not in the documents

Result 1  (similarity: -0.0623)
  Metadata : {'chunk_index': 5, 'source': 'product_faq'}
  Text     : Mixed-language queries are partially supported. Q: What happens if the answer is not in our documents? A: The system is designed to say it does not kn...

Result 2  (similarity: -0.3118)
  Metadata : {'chunk_index': 1, 'source': 'product_faq'}
  Text     : Q: How is the AI trained on our data? A: We use a technique called Retrieval-Augmented Generation. Your sales playbooks, product documents, and case s...



### Query 4: An Out-of-Domain Query

In [10]:
# A query about something not in the documents
results = retrieve("how do I bake a chocolate cake", n_results=3)

print("Query: how do I bake a chocolate cake")
print()
print_chroma_results(results)
print("Note: the distances for an out-of-domain query are much higher.")
print("In a production system, you would set a distance threshold and return")
print("'not found' if the best result is above that threshold.")

Query: how do I bake a chocolate cake

Result 1  (similarity: -0.8828)
  Metadata : {'chunk_index': 1, 'source': 'sales_notes'}
  Text     : The most effective technique is to reframe from price to ROI. If your product saves a team 10 hours per week at 500 rupees per hour, that is 5000 rupe...

Result 2  (similarity: -0.9433)
  Metadata : {'chunk_index': 0, 'source': 'product_faq'}
  Text     : PRODUCT FAQ — AI SALES ASSISTANT TOOL

Q: What does the AI Sales Assistant do? A: It helps sales teams prepare for calls, handle objections in real ti...

Result 3  (similarity: -0.951)
  Metadata : {'chunk_index': 3, 'source': 'sales_notes'}
  Text     : Situation: "What does your current process for handling invoicing look like?"
Problem: "Where does that process break down most often?"
Implication: "...

Note: the distances for an out-of-domain query are much higher.
In a production system, you would set a distance threshold and return
'not found' if the best result is above that threshold.




## Inspecting the Full Pipeline

Let's trace exactly what happened when we ran Query 1, step by step.

In [11]:
query = "how to handle price objections"

# Step 1: Embed the query
q_vec = embed(query)
print("Step 1 — Query embedding")
print(f"  Shape: {q_vec.shape}")
print(f"  First 5 values: {q_vec[:5].round(4)}")
print()

# Step 2: Search ChromaDB
results = collection.query(query_embeddings=[q_vec.tolist()], n_results=3)
print("Step 2 — ChromaDB returned")
print(f"  {len(results['documents'][0])} results")
print()

# Step 3: Show what was retrieved
print("Step 3 — Retrieved chunks:")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"  [{i+1}] Source: {meta['source']} | Distance: {dist:.4f}")
    print(f"       {doc[:120]}..." if len(doc) > 120 else f"       {doc}")
    print()

Step 1 — Query embedding
  Shape: (384,)
  First 5 values: [-0.0315  0.1461 -0.0214 -0.0084  0.0096]

Step 2 — ChromaDB returned
  3 results

Step 3 — Retrieved chunks:
  [1] Source: sales_notes | Distance: 0.3495
       SALES TRAINING NOTES — HANDLING OBJECTIONS AND CLOSING DEALS

Section 1: Price Objections

When a prospect says your pro...

  [2] Source: sales_notes | Distance: 1.1825
       Section 4: Handling Competition

When a prospect mentions a competitor, do not attack the competitor. It looks desperate...

  [3] Source: sales_notes | Distance: 1.1965
       The most effective technique is to reframe from price to ROI. If your product saves a team 10 hours per week at 500 rupe...




## What Comes Next

In the next class, we will add the Generation step.

The retrieved chunks will be formatted into a prompt like this:

```
Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
[chunk 1]
[chunk 2]
[chunk 3]

Question: how to handle price objections
```

That prompt is sent to an LLM (Gemini, GPT, or a local model via Ollama).
The LLM reads the context and generates an answer grounded in your actual documents.

That is the complete RAG pipeline.


## Summary

- Phase 1 (Ingestion): load -> chunk -> embed -> store. Run once.
- Phase 2 (Retrieval): embed query -> search vector DB -> return top-k chunks. Run on every query.
- The quality of retrieval depends on chunk size, chunking strategy, and embedding model.
- An out-of-domain query returns results with high distance scores — use a threshold to detect this.
- Metadata filtering lets you scope search to specific documents or categories.